# Fleet v0.8 Native FleetForCausalLM — Option A

Set the runtime to a GPU.

For automatic upload, put a **new/rotated** Hugging Face write token in
Colab Secrets as `HF_TOKEN`.

Upload `fleet-semantic-v0.8-native.zip` in Cell 1.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!rm -rf /content/fleet-v0.8
!mkdir -p /content/fleet-v0.8
!unzip -q /content/fleet-semantic-v0.8-native.zip -d /content/fleet-v0.8
%cd /content/fleet-v0.8
!pip install -q -U pip
!pip install -q -e ".[server]"

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
!fleet build \
  --config configs/experts.json \
  --output artifacts/fleet-bank

In [ ]:
!fleet semantic-build \
  --bank artifacts/fleet-bank \
  --anchors configs/semantic_anchors.json

In [ ]:
!fleet validate artifacts/fleet-bank

## Build the standalone checkpoint

This downloads the full Qwen3-4B-Instruct-2507 backbone and embeds it into the
Fleet checkpoint. Expect several GB of output. The checkpoint is saved in
sharded safetensors.

In [ ]:
!fleet native-build \
  --bank artifacts/fleet-bank \
  --output artifacts/Fleet-Semantic-4B \
  --dtype float16 \
  --max-shard-size 4GB

In [ ]:
!find artifacts/Fleet-Semantic-4B -maxdepth 1 -type f -printf "%f\n" | sort

## Mandatory round-trip

This reloads the just-built checkpoint through Transformers AutoModel and verifies
the class is actually `FleetForCausalLM`.

In [ ]:
!fleet native-validate \
  --model artifacts/Fleet-Semantic-4B

## Automatic Hub release

The command below rebuilds + validates before upload, so it is deliberately redundant.
It will not upload if standalone reload/generation fails.

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
!fleet native-release \
  --bank artifacts/fleet-bank \
  --output artifacts/Fleet-Semantic-4B \
  --repo noumenon-labs/Fleet-Semantic-4B \
  --max-shard-size 4GB \
  --num-workers 4